In [1]:
%pip install torch transformers

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import BertTokenizer
from transformers import BertForSequenceClassification
from transformers import Trainer, TrainingArguments

import sys
import os

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from data.DataLoader import DataLoader
from data.Elastic import Elastic

In [3]:
dataloader = DataLoader(
    Elastic(timeout=5), 
    index='lexical'
)

In [4]:
from torch.utils.data import Dataset, DataLoader
import torch

In [5]:
urls = dataloader.df['url'].tolist()
labels = dataloader.df['type'].tolist()

In [6]:
# Using a pre-trained tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize URLs
def tokenize_urls(urls):
    return tokenizer(urls, padding=True, truncation=True, max_length=512, return_tensors='pt')

class PhishingDataset(Dataset):
    def __init__(self, encodings, labels):
        # Encodings are expected to be a dict where values are already tensors
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        # Directly use the tensor slices without re-wrapping them into new tensors
        item = {key: val[idx] for key, val in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

    def __len__(self):
        return self.labels.size(0)

# Assume URLs and labels are already defined above
encoded_inputs = tokenize_urls(urls)

# Convert the labels to a tensor outside of the dataset initialization to ensure it is properly managed
labels_tensor = torch.tensor(labels, dtype=torch.long)

# Create the dataset
dataset = PhishingDataset(encoded_inputs, labels_tensor)

In [7]:
# Load pre-trained BERT model for sequence classification
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

# Freeze all layers except the classifier to speed up training (optional)
for name, param in model.named_parameters():
    if 'classifier' not in name:  # Freeze layers other than the classifier
        param.requires_grad = False

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.bias', 'cls.seq_relationship.weight', 'cls.predictions.decoder.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at

In [ ]:
training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=8,   # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
)

trainer = Trainer(
    model=model,                         # the instantiated Transformers model to be trained
    args=training_args,                  # training arguments, defined above
    train_dataset=dataset,         # training dataset
)

trainer.train()

In [ ]:
trainer.evaluate()